# Lab 4: From Counts to Dependence Structures

In this lab you will look at one dataset, ask **which variables depend on which others**, and draw a graph that summarises what you found. The graph is just a picture of the data: it tells us which proportions factorise and which do not.

**Data:** a table of 10 000 made-up travellers, each described by four categorical variables:

- `financial_status` — Wealthy / Middle income / Limited means
- `ticket_class` — First / Second / Third
- `purpose` — Business / Leisure / Migration
- `party` — Solo / Couple / Family

Each group has a slightly different dataset (`group_A.csv` … `group_H.csv`). You also share one extra file, `collider_upgrade.csv`.


## Learning objectives

After this lab you can:

1. Read a categorical dataset as a set of counts and proportions.
2. Decide whether two categorical variables are associated, using both plots and a statistical test.
3. Use **conditioning** to see whether an association disappears or appears.
4. Build a graph of variables from conditional-independence facts.
5. Explain what the graph can and cannot tell us (direction of edges, hidden variables, sample size).


In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())

# Setup: run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The helper functions live in lab_helpers.py in the same folder as this notebook.
from lab_helpers import (
    load_group, counts, cond_props, assoc_test,
    bubble_panels, diagonal, sample_sweep, format_p,
    F, C, P, T
)

# TODO: replace with the letter your instructor assigned to your group.
MY_GROUP = "A"

df = load_group(MY_GROUP)
print(f"Loaded {len(df):,} rows for group {MY_GROUP}")
df.head()


---
## Module A: One variable at a time

Before looking at relationships, get familiar with the marginal counts.


In [ ]:
# TODO: How many travellers are there in each ticket class?
# Hint: value_counts() respects the categorical order defined in lab_helpers.
counts_ticket = df[C].value_counts()
props_ticket = df[C].value_counts(normalize=True)

pd.DataFrame({'count': counts_ticket, 'proportion': props_ticket.round(3)})


In [ ]:
# TODO: Plot the marginal distribution of one variable of your choice.
variable_to_plot = C   # change to F, P, or T if you like
df[variable_to_plot].value_counts().plot(kind='bar', rot=0, color='steelblue', edgecolor='black')
plt.title(f'Distribution of {variable_to_plot}')
plt.ylabel('count')
plt.show()


**Your answer:** Which level is most common? Does the distribution look roughly uniform or very uneven?

*Write 1–2 sentences.*


---
## Module B: Pairwise associations

Two variables are **associated** if knowing one changes the proportions of the other. We will look at counts, conditional-proportion plots, and a numerical test.


In [ ]:
# TODO: Cross-tabulate ticket_class and party.
ct = counts(df, C, T)
ct


In [ ]:
# TODO: Turn the table above into row proportions P(party | ticket_class).
cond_props(df, C, T)


In [ ]:
# Visualise the same conditional proportions as a bubble plot.
# Larger bubbles mean a higher conditional probability.
fig = bubble_panels(df, C, T)
plt.show()


In [ ]:
# TODO: Run an association test for ticket_class vs party.
r = assoc_test(df, C, T)
print(f"Cramer's V = {r['V']:.3f}")
print(f"chi2 statistic = {r['stat']:.2f}, p = {format_p(r['p'])}, n = {r['n']}")


Cramer's V is an effect size: 0 means no association, larger values mean stronger association. The p-value tells us whether the association is clearly visible in *this* sample.

**Your answer:** In your table, does the distribution of `party` look about the same for every ticket class, or does it change? How do you read that from the bubble plot?


In [ ]:
# TODO: Test all 6 unordered pairs of variables.
# Fill the results into your worksheet (or the markdown cell below).
from itertools import combinations

pairs = list(combinations([F, C, P, T], 2))
rows = []
for a, b in pairs:
    r = assoc_test(df, a, b)
    rows.append({'pair': f'{a} - {b}', 'V': round(r['V'], 3), 'p': format_p(r['p'])})
pd.DataFrame(rows)


**Your answer:** Which pairs look clearly associated (small p, larger V) and which look weak or absent? Are there any surprises?

| pair | associated? (yes / no / borderline) | notes |
|---|---|---|
| financial_status - ticket_class | | |
| financial_status - purpose | | |
| financial_status - party | | |
| ticket_class - purpose | | |
| ticket_class - party | | |
| purpose - party | | |


---
## Module C: Conditioning

A pair can be associated *overall* but independent once we look inside a subgroup. That is the key idea for building a graph.

We say: **X is independent of Y given Z** if, inside every level of Z, knowing X does not change the proportions of Y.


In [ ]:
# TODO: Focus on ticket_class and party.
# Are they still associated if we look only at travellers with the same purpose?
fig = bubble_panels(df, C, T, given=P)
plt.show()


In [ ]:
# TODO: Run the association test again, this time conditioning on purpose.
r_cond = assoc_test(df, C, T, given=P)
print(f"C-T | purpose: V={r_cond['V']:.3f}, p={format_p(r_cond['p'])}")


**Your answer:** Did the p-value change? What does that mean in plain language?

*Write 2–3 sentences.*


In [ ]:
# TODO: Try conditioning on each of the other variables for the pair ticket_class - party.
for z in [F, P]:
    r = assoc_test(df, C, T, given=z)
    print(f'C - T | {z}: p = {format_p(r["p"])}, V = {r["V"]:.3f}')


### Conditioning table

For each pair, test independence while conditioning on a third variable. Mark **I** if the pair looks independent after conditioning (p reasonably large), **D** if it still looks dependent (p small), and **~** if the answer is noisy because the subgroup is small.

| pair | condition on F | condition on C | condition on P | condition on T |
|---|---|---|---|---|
| F - C | — | | | |
| F - P | | — | | |
| F - T | | | | |
| C - P | | — | | |
| C - T | | | | |
| P - T | | | | — |

*For the pair's own variables the cell is left blank.*


In [ ]:
# TODO: Fill the conditioning table using assoc_test(..., given=...).
# Example for one cell:
print('F - T given P:', format_p(assoc_test(df, F, T, given=P)['p']))


**Your answer:** Which pairs stay associated no matter what you condition on? Which pairs become independent once you condition on the right variable?

*Use the table above to decide which variables should be connected by an edge in your graph.*


### Optional visual check: diagonal plot

For a pair that becomes independent after conditioning, the diagonal plot is a quick way to see it. If all the points lie on the diagonal, the two rows have the same proportions.


In [ ]:
# TODO: Diagonal plot for a pair that you think is independent after conditioning.
# Example: P(party | ticket_class=First) vs P(party | ticket_class=Third),
# coloured by purpose.
ax = diagonal(df, C, T, lo='First', hi='Third', given=P)
plt.show()


---
## Module D: Noise, sample size, and what the data cannot decide

Real samples are noisy. A non-significant p-value is not proof of independence, and a significant p-value may be tiny in a very large sample. We explore the effect of sample size on the ticket_class - party association, after conditioning on purpose.


In [ ]:
# TODO: Re-run the test on smaller random subsamples.
sweep = sample_sweep(df, C, T, given=P, sizes=(200, 500, 1000, 5000), n_rep=30)
sweep.groupby('n')['p'].median()


In [ ]:
# TODO: Plot p-value against sample size.
sizes_used = sorted(sweep['n'].unique())
plt.figure(figsize=(8, 4))
plt.boxplot([sweep[sweep['n'] == n]['p'] for n in sizes_used], tick_labels=sizes_used)
plt.axhline(0.05, color='red', linestyle='--', label='p = 0.05')
plt.xlabel('sample size n')
plt.ylabel('p-value')
plt.title('p-value of C-T association | purpose vs sample size')
plt.legend()
plt.show()


**Your answer:** With only 200 travellers, would you trust a non-significant p-value as evidence of independence? With 10 000?

*Write 2–3 sentences.*


In [ ]:
# TODO: Run a permutation test for C - T | purpose.
# The permutation test does not rely on the chi-square approximation.
r_perm = assoc_test(df, C, T, given=P, method='perm', n_perm=499, seed=42)
print(f"Permutation p = {format_p(r_perm['p'])}")


---
## Module E: A collider — when conditioning creates association

So far conditioning has removed associations. In the shared `collider_upgrade.csv` file the opposite happens: two variables are *marginally* independent, but become associated once we look only at travellers who were upgraded (or only at those who were not).

Variables:
- `loyalty_member` — Yes / No
- `peak_season` — Peak / Off-peak
- `upgraded` — Yes / No


In [ ]:
# Load the collider dataset.
up = pd.read_csv('data/collider_upgrade.csv')
up.head()


In [ ]:
# TODO: Are loyalty_member and peak_season associated in the whole table?
r_marg = assoc_test(up, 'loyalty_member', 'peak_season')
print(f"loyalty - peak_season: V={r_marg['V']:.3f}, p={format_p(r_marg['p'])}")


In [ ]:
# TODO: Now condition on upgraded = Yes. Are they associated inside that subgroup?
up_yes = up[up['upgraded'] == 'Yes']
r_yes = assoc_test(up_yes, 'loyalty_member', 'peak_season')
print(f"loyalty - peak_season | upgraded=Yes: V={r_yes['V']:.3f}, p={format_p(r_yes['p'])}")

# TODO: What about upgraded = No?
up_no = up[up['upgraded'] == 'No']
r_no = assoc_test(up_no, 'loyalty_member', 'peak_season')
print(f"loyalty - peak_season | upgraded=No:  V={r_no['V']:.3f}, p={format_p(r_no['p'])}")


**Your answer:** Why might two independent causes become associated once we look only at the outcome they both influence? Think of it as a filter: who gets upgraded?

*Write 3–4 sentences.*


---
## Module F: Build the graph

A graph of variables has one node per variable and an edge between two nodes if they stay associated after conditioning on the other variables.

The graph is *not* a claim about causation. It is a compact summary of the dependence structure in the data.


In [ ]:
# TODO: List the edges of your graph as (variable1, variable2) pairs.
# Use the conditioning table from Module C.
my_edges = [
    # Example (replace with your decisions):
    # (F, C),
    # (F, P),
    # (P, T),
]
print('My graph edges:', my_edges)


**Draw your graph here (on paper or with a small sketch):**

- Draw one circle per variable.
- Draw an edge between any two variables that stay associated after conditioning.
- Leave out edges for pairs that become independent after conditioning.


**Your answer:** What can your graph *not* tell you? For example, can it tell you which variable causes which? Can it tell you whether an unmeasured variable might explain an edge?

*Write 3–5 sentences.*


---
## Exit ticket

Answer in your own words (2–3 sentences each):

1. What is the difference between a marginal association and a conditional association?
2. When is a graph of variables a useful summary of a dataset?
3. Why do we need to be careful with p-values when the sample is small?


---
## Appendix: variable dictionary

| variable | levels | meaning |
|---|---|---|
| `financial_status` | Wealthy, Middle income, Limited means | proxy for overall resources |
| `ticket_class` | First, Second, Third | ticket class purchased |
| `purpose` | Business, Leisure, Migration | reason for travelling |
| `party` | Solo, Couple, Family | composition of travelling party |
